# Topic Modeling & NER: By Crisis Window

Runs the same `src/topic_modeling.py` / `src/ner.py` pipeline as `02_topic_model_ner_overall.ipynb`, once per crisis window from `src/crisis_windows.py`, so the two notebooks can't drift apart on method even though they differ on which subset of posts they run over.

**Windows are DRAFT** -- `src/crisis_windows.py` has the current placeholder dates from Project_Proposal.md §4; update that one file once verified and both this notebook and any other window-based analysis pick up the fix automatically. **Windows overlap by design** (a post can belong to Ukraine/energy AND Gaza AND cost-of-living at once) -- see that module's docstring.

**Not executed here** -- same GPU-server requirement as notebook 03.

## Setup

In [ ]:
pip install de_core_news_lg

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sshfs
import yaml
from src import plot_style
from src.topic_modeling import prepare_topic_docs, fit_topic_model_two_pass
from src.ner import extract_entities, entity_counts_over_time
from src.crisis_windows import CRISIS_WINDOWS, resolve_windows, filter_by_window

plot_style.set_style()

# data server connection
#with open("C:/Users/Elena Solar/.ssh/sshfs_config.yaml") as f:
#    config = yaml.safe_load(f)
    
with open("../../login/sshfs_config.yaml") as f:
    config = yaml.safe_load(f)

fs = sshfs.SSHFileSystem(
    host=config["sshfs"]["host"],
    username=config["sshfs"]["username"],
    client_keys=config["sshfs"]["client_keys"],
)


## Load Data

In [ ]:
with fs.open("/data/workspace/esolar/FFF_Climate_Framing/data/raw/FFF_german_01-2015_12-2022.csv") as f:
    df_1 = pd.read_csv(f, low_memory = False)

with fs.open("/data/workspace/esolar/FFF_Climate_Framing/data/raw/FFF_german_01-2023_07-2026.csv") as f:
    df_2 = pd.read_csv(f, low_memory = False)

df = pd.concat([df_1, df_2]).sort_values(by="creation_time").reset_index(drop=True)
df["creation_time"] = pd.to_datetime(df["creation_time"])
df = df[df["post_owner.username"] != "hackhermannjosef"].reset_index(drop=True)

df_FFF = pd.read_csv("../data/processed/FFF_german.csv")
df_FFF["creation_time"] = pd.to_datetime(df_FFF["creation_time"], utc=True)
df_FFF

print(f"{len(df_FFF)} posts from official FFF accounts")

In [ ]:
df_FFF

## Resolve Windows Against This Corpus

In [ ]:
resolved_windows = resolve_windows(CRISIS_WINDOWS, df_FFF["creation_time"].max())
for name, start, end, description in resolved_windows:
    n = len(filter_by_window(df_FFF, (name, start, end, description)))
    print(f"{name:16s} {start.date()} to {end.date()}: {n} posts -- {description}")

## Topic Modeling & NER Per Window

Two-pass fit per window (`fit_topic_model_two_pass()`, same function as `02_topic_model_ner_overall.ipynb`) -- broad diagnostic pass, then a refined re-fit -- but aimed at a smaller topic count than the overall corpus's 30, since these subsets are much smaller (as little as ~500 posts vs. ~4,100 overall). `target_n_topics` scales down proportionally to each window's size (floor of 8). `refined_min_cluster_size` uses a coarser ~2%-of-window heuristic (floor of 10) rather than the overall's ~1% -- the flatter ~1% rule floors out for windows this small and ends up barely different from the broad pass's own `min_cluster_size=10`, i.e. no real consolidation. Both are starting points, not guarantees (no closed-form relationship between `min_cluster_size` and resulting topic count) -- check the printed counts per window and retune if one looks off.

In [ ]:
OVERALL_TARGET_N_TOPICS = 30  # must match 02_topic_model_ner_overall.ipynb's TARGET_N_TOPICS to scale proportionally
OVERALL_N_DOCS = len(df_FFF)

window_results = {}

for name, start, end, description in resolved_windows:
    window_df = filter_by_window(df_FFF, (name, start, end, description))
    topic_docs, topic_dates, _non_empty_mask = prepare_topic_docs(window_df)

    target_n_topics = max(8, round(OVERALL_TARGET_N_TOPICS * len(topic_docs) / OVERALL_N_DOCS))
    refined_min_cluster_size = max(10, len(topic_docs) // 50)

    fit_result = fit_topic_model_two_pass(
        topic_docs.tolist(), target_n_topics=target_n_topics, refined_min_cluster_size=refined_min_cluster_size,
    )
    topic_model, topics, topic_info = fit_result["topic_model"], fit_result["topics"], fit_result["topic_info"]

    ner_ids = window_df["id"].astype(str) if "id" in window_df.columns else window_df.index.astype(str)
    entities = extract_entities(window_df["text"].tolist(), ner_ids.tolist())

    window_results[name] = {
        "topic_model": topic_model,
        "topics": topics,
        "topic_info": topic_info,
        "n_posts": len(window_df),
        "n_docs_scored": len(topic_docs),
        "n_outliers": sum(1 for t in topics if t == -1),
        "outlier_frac": fit_result["refined_outlier_frac"],
        "target_n_topics": target_n_topics,
        "min_cluster_size": refined_min_cluster_size,
        "broad_n_topics": fit_result["broad_n_topics"],
        "entities": entities,
    }
    print(f"{name:16s}: {len(topic_docs)} docs -- broad {fit_result['broad_n_topics']} topics -> "
          f"refined (min_cluster_size={refined_min_cluster_size}, target {target_n_topics}) "
          f"{(topic_info['Topic'] != -1).sum()} topics, {fit_result['refined_outlier_frac'] * 100:.1f}% outliers")

## Cross-Window Summary

In [ ]:
summary = pd.DataFrame([
    {
        "window": name,
        "n_posts": r["n_posts"],
        "n_topics": (r["topic_info"]["Topic"] != -1).sum(),
        "outlier_pct": r["outlier_frac"] * 100,
        "top_entity": r["entities"]["entity_text"].value_counts().idxmax() if len(r["entities"]) else None,
    }
    for name, r in window_results.items()
])
summary

In [ ]:
fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_SINGLE)
sns.barplot(data=summary, x="window", y="n_topics", ax=ax)
plot_style.annotate_bars(ax)
ax.set_title("Topics Found Per Crisis Window")
ax.set_xlabel("Window")
ax.set_ylabel("Number of Topics")
ax.tick_params(axis="x", rotation=30)
#plot_style.savefig(fig, "FFF/by_window_topic_counts")
plt.show()

### Topics by Size, Per Window

Small multiples -- one panel per crisis window, topics sorted by post count within that window.

In [ ]:
fig, axes = plt.subplots(1, len(window_results), figsize=(4 * len(window_results), 5), sharex=False)

for ax, (name, r) in zip(axes, window_results.items()):
    window_topics = r["topic_info"][r["topic_info"]["Topic"] != -1].sort_values("Count", ascending=False)
    sns.barplot(data=window_topics, y="Name", x="Count", order=window_topics["Name"], ax=ax)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Posts")
    ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=7)

fig.suptitle("Topics by Size, Per Crisis Window")
fig.tight_layout()
#plot_style.savefig(fig, "FFF/by_window_topic_sizes")
plt.show()

In [ ]:
window_results.keys()

In [ ]:
window_results["cost_of_living"]['topic_info']

## Save Results

In [ ]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

for name, r in window_results.items():
    r["entities"].to_csv(output_dir / f"window_{name}_entities.csv", index=False)

summary.to_csv(output_dir / "by_window_topic_summary.csv", index=False)
print(f"saved per-window entity tables and summary to {output_dir}")